# 01 · Understanding the data: what does a microservice failure look like?

**Goal of this project:** when something breaks in a system made of many small services, find **which service caused it** (root-cause *localisation*) and **what kind of fault** it is. This is the "root-cause analysis" step of incident management, done on real telemetry.

### The dataset: RCAEval, suite RE2
RCAEval (WWW 2025 benchmark) ran three open-source demo shops/apps, deliberately broke one service at a time and recorded everything:

| System | Short name | Services with metrics | What it is |
|---|---|---|---|
| Online Boutique | `ob` | 12 | Google's e-commerce demo (Go, Python, Java, …) |
| Sock Shop | `ss` | 15 | Weaveworks' sock shop |
| Train Ticket | `tt` | 68 | a large ticket-booking system (Java) |

**Fault types (6):** CPU stress, memory stress, disk stress, network delay, packet loss, socket stress.

**One case** = 12 minutes of normal operation → fault injected at `inject_time` → 12 more minutes. We get:
- **metrics** (1 row per second; CPU, memory, disk I/O, sockets, workload, errors, latency per service),
- **logs** (every log line with its container),
- **traces** (every request "span": which service, which operation, how long it took). Not recorded for Sock Shop.

The **label** (`root_cause_service`) is known because the fault was injected on purpose, a real ground truth, unlike most incident datasets.

> **Why this matters:** in a bank's operations centre an alert says "checkout is slow", but the cause is often a *different* service. Automatically ranking the likely culprits saves the on-call engineer time.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*(mismatched devices|pandas version 4).*")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "reports" / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

from rcalens.data import case_index, load_case, split_metric
idx = case_index()
print(len(idx), "cases")
idx.groupby("system").agg(cases=("case", "size"), root_cause_services=("root_cause_service", "nunique"),
                          fault_types=("fault", "nunique"), metrics=("n_metrics", "median"),
                          log_lines=("n_logs", "median"), spans=("n_traces", "median"))

**Reading the table:** 90 cases per system = 5 root-cause services × 6 fault types × 3 repetitions. Train Ticket is much bigger (360+ metric columns, ~1M spans per case). Sock Shop has no traces (0 spans).

In [ ]:
pd.crosstab([idx["system"], idx["root_cause_service"]], idx["fault"])

Every (service, fault) pair appears exactly 3 times. **Important consequence for evaluation:** the 3 repetitions are near-copies of each other. If one repetition is in training and another in test, the model can "recognise" the case instead of reasoning about it. So our splits always keep all 3 repetitions together (notebook 04).

## What the model must choose from
For each case the candidate list is **every service that has metrics**, not just the 5 that the benchmark happens to break (using that list would be cheating: in real life you don't know which 5 services will fail).

In [ ]:
cand = {s: len(load_case(idx.loc[idx.system == s, "case"].iloc[0], traces=False).services) for s in ["ob", "ss", "tt"]}
rand = {s: {"candidates": n, "random top-1": 1 / n, "random top-3": min(3, n) / n} for s, n in cand.items()}
pd.DataFrame(rand).T.round(3)

A random guess finds the culprit first time 8% (Online Boutique), 7% (Sock Shop) or **1.5%** (Train Ticket) of the time. That's the floor.

## Look at one failure: a CPU fault on `checkoutservice`

In [ ]:
c = load_case("re2ob_checkoutservice_cpu_1")
m = c.metrics.copy(); m["minute"] = (m["time"] - c.inject_time) / 60
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharex=True)
for ax, typ in zip(axes, ["cpu", "mem", "latency-90"]):
    for col in [k for k in m.columns if k.endswith("_" + typ)]:
        svc = split_metric(col)[0]
        ax.plot(m["minute"], m[col], lw=2.2 if svc == "checkoutservice" else 0.7,
                color="#c0392b" if svc == "checkoutservice" else "#9aa7b4", label="checkoutservice" if svc == "checkoutservice" else None)
    ax.axvline(0, color="k", ls="--", lw=1); ax.set_title(typ); ax.set_xlabel("minutes from fault injection")
axes[0].legend(); plt.tight_layout(); plt.savefig(FIG / "example_cpu_fault.png", dpi=120); plt.show()

The red line (the root cause) jumps in CPU right at the injection (dashed line). Other services barely move. **But it is not always this clean:**
- a *delay* or *loss* fault on one service shows up as **higher latency in its callers** too (the slowness travels upstream),
- some metrics are noisy or flat (a service that normally has 0 errors makes any error look "infinitely" anomalous),
- Train Ticket has 68 services, so there are many chances for an innocent service to look odd.

That is why we compute several kinds of evidence per service and let a model learn how to weigh them.

## Logs and traces around the injection

In [ ]:
lg = c.logs.copy(); lg["minute"] = (lg["timestamp"].astype("int64") - c.inject_time) // 60
per_min = lg.groupby(["minute", "container_name"]).size().unstack(fill_value=0)
ax = per_min[per_min.sum().sort_values(ascending=False).index[:6]].plot(figsize=(10, 3.4), lw=1.2)
ax.axvline(0, color="k", ls="--"); ax.set_ylabel("log lines / minute"); ax.set_title("log volume per container"); plt.tight_layout(); plt.show()
if c.traces is not None:
    t = c.traces.copy(); t["minute"] = (t["startTimeMillis"].astype("int64") // 1000 - c.inject_time) // 60
    p95 = t.groupby(["minute", "serviceName"])["duration"].quantile(0.95).unstack()
    ax = (p95 / 1000).plot(figsize=(10, 3.4), lw=1.2, logy=True)
    ax.axvline(0, color="k", ls="--"); ax.set_ylabel("p95 span duration (ms)"); ax.set_title("trace latency per service"); plt.tight_layout(); plt.show()

### Takeaways
1. Each case is a **ranking problem**: order 12–68 candidate services so the true culprit comes first.
2. Evidence comes from three sources (metrics, logs, traces). Traces are missing for Sock Shop, so models must cope with missing data.
3. The 3 repetitions per (service, fault) must stay together in any train/test split.

➡️ Next: **02_feature_engineering**: turn each service's behaviour before/after the fault into numbers.